In [1]:
import pandas as pd
from IPython.display import display

In [2]:
df=pd.read_csv("../docs/forensics/seed-data.csv")


In [3]:
print("columns=", df.columns.tolist())
print("rows_total=", len(df))
display(df.head(5))
display(df['label'].value_counts().head(5))


columns= ['date_utc', 'block_number', 'tx_hash', 'from_address', 'to_address', 'value_or_token', 'label', 'source']
rows_total= 15


,date_utc,block_number,tx_hash,from_address,to_address,value_or_token,label,source
0,2022-04-17T12:24:16Z,14602790,0xcd314668aaa9bbfebaf1a0bd2b6553d01dd58899c508...,0x1c5dCdd006EA78a7E4783f9e6021C32935a10fb4,0x79224bc0bf70ec34f0ef56ed8251619499a59def,0 ETH,exploit_tx,https://etherscan.io/tx/0xcd314668aaa9bbfebaf1...
1,2022-04-17T14:56:04Z,14603456,0x3bc6c39d2579517f37c9ee20ac3345c99447a5464e06...,0x1c5dCdd006EA78a7E4783f9e6021C32935a10fb4,0xd90e2f925DA726b50C4Ed8D0Fb90Ad053324F31b,100 ETH,deposit_tornado_cash,https://etherscan.io/tx/0x3bc6c39d2579517f37c9...
2,2022-04-17T14:55:51Z,14603455,0xe8b60dc187cf44e28516774865d8c9bcef88a8b36079...,0x1c5dCdd006EA78a7E4783f9e6021C32935a10fb4,0xd90e2f925DA726b50C4Ed8D0Fb90Ad053324F31b,100 ETH,deposit_tornado_02,https://etherscan.io/tx/0xe8b60dc187cf44e28516...
3,2022-04-17T14:55:25Z,14603453,0xa5e1b63d8195d0c3302d922b6ec8abb64f0165021dd1...,0x1c5dCdd006EA78a7E4783f9e6021C32935a10fb4,0xd90e2f925DA726b50C4Ed8D0Fb90Ad053324F31b,100 ETH,deposit_tornado_03,https://etherscan.io/tx/0xa5e1b63d8195d0c3302d...
4,2022-04-17T14:54:59Z,14603451,0x557dab03a7046fc6280416cb8c2077cbbf5836a91bce...,0x1c5dCdd006EA78a7E4783f9e6021C32935a10fb4,0xd90e2f925DA726b50C4Ed8D0Fb90Ad053324F31b,100 ETH,deposit_tornado_04,https://etherscan.io/tx/0x557dab03a7046fc62804...


label
exploit_tx              1
deposit_tornado_cash    1
deposit_tornado_02      1
deposit_tornado_03      1
deposit_tornado_04      1
Name: count, dtype: int64

# Timeline v0

In [4]:
dt = pd.to_datetime(df["date_utc"], errors="coerce", utc=True)
date_ok = dt.notna()
with_date = df[date_ok].assign(_dt=dt[date_ok]).sort_values('_dt')
no_date = df[~date_ok].sort_values("block_number")
df_sorted = pd.concat([with_date, no_date], ignore_index=True).drop(columns=["_dt"], errors="ignore")

df_sorted[["date_utc","block_number","label"]].head(12)

,date_utc,block_number,label
0,2022-04-17T12:24:16Z,14602790,exploit_tx
1,2022-04-17T14:51:09Z,14603434,deposit_tornado_13
2,2022-04-17T14:51:18Z,14603432,deposit_tornado_14
3,2022-04-17T14:51:24Z,14603436,deposit_tornado_12
4,2022-04-17T14:51:39Z,14603438,deposit_tornado_11
5,2022-04-17T14:52:05Z,14603439,deposit_tornado_10
6,2022-04-17T14:52:19Z,14603440,deposit_tornado_09
7,2022-04-17T14:53:07Z,14603443,deposit_tornado_08
8,2022-04-17T14:53:50Z,14603446,deposit_tornado_07
9,2022-04-17T14:54:20Z,14603448,deposit_tornado_06


In [5]:
#timeline_table = 
s_tx_hash = df_sorted['tx_hash'].str.slice(0, 10).astype(str) + '...'
s_from_address = df_sorted['from_address'].str.slice(0, 10).astype(str) + '...'
s_to_address = df_sorted['to_address'].str.slice(0, 10).astype(str) + '...'

timeline_table = pd.concat(
    [df_sorted['date_utc'], 
    df_sorted['label'],
    s_tx_hash,
    s_from_address,
    s_to_address,
    df_sorted['value_or_token']],
    axis=1)

timeline_table.head(8)

,date_utc,label,tx_hash,from_address,to_address,value_or_token
0,2022-04-17T12:24:16Z,exploit_tx,0xcd314668...,0x1c5dCdd0...,0x79224bc0...,0 ETH
1,2022-04-17T14:51:09Z,deposit_tornado_13,0x91f0edde...,0x1c5dCdd0...,0xd90e2f92...,100 ETH
2,2022-04-17T14:51:18Z,deposit_tornado_14,0xe1c44898...,0x1c5dCdd0...,0xd90e2f92...,100 ETH
3,2022-04-17T14:51:24Z,deposit_tornado_12,0xb04037d8...,0x1c5dCdd0...,0xd90e2f92...,100 ETH
4,2022-04-17T14:51:39Z,deposit_tornado_11,0x1dfd7473...,0x1c5dCdd0...,0xd90e2f92...,100 ETH
5,2022-04-17T14:52:05Z,deposit_tornado_10,0xe3e1333b...,0x1c5dCdd0...,0xd90e2f92...,100 ETH
6,2022-04-17T14:52:19Z,deposit_tornado_09,0x16f64a56...,0x1c5dCdd0...,0xd90e2f92...,100 ETH
7,2022-04-17T14:53:07Z,deposit_tornado_08,0xe49cf442...,0x1c5dCdd0...,0xd90e2f92...,100 ETH


## Timeline v0

- [2022-04-17T12:24:16Z] Exploit transaction is recorded (label=exploit_tx, tx=0xcd314668..., value=0 ETH)
- [2022-04-17T14:51:09Z] Tornado Cash deposit occurs (Timeline deposit 1) (label=deposit_tornado_13, tx=0x91f0edde..., value=100 ETH)
- [2022-04-17T14:51:18Z] Tornado Cash deposit occurs (Timeline deposit 2) (label=deposit_tornado_14, tx=0xe1c44898..., value=100 ETH)
- [2022-04-17T14:51:24Z] Tornado Cash deposit occurs (Timeline deposit 3) (label=deposit_tornado_12, tx=0xb04037d8..., value=100 ETH)
- [2022-04-17T14:51:39Z] Tornado Cash deposit occurs (Timeline deposit 4) (label=deposit_tornado_11, tx=0x1dfd7473..., value=100 ETH)
- [2022-04-17T14:52:05Z] Tornado Cash deposit occurs (Timeline deposit 5) (label=deposit_tornado_10, tx=0xe3e1333b..., value=100 ETH)
- [2022-04-17T14:52:19Z] Tornado Cash deposit occurs (Timeline deposit 6) (label=deposit_tornado_09, tx=0x16f64a56..., value=100 ETH)
- [2022-04-17T14:53:07Z] Tornado Cash deposit occurs (Timeline deposit 7) (label=deposit_tornado_08, tx=0xe49cf442..., value=100 ETH)

## Flow Map v0

In [6]:
colums_to_copy = ['from_address', 'to_address', 'tx_hash', 'label', 'value_or_token']
edges = df[colums_to_copy].copy()
edges['tx_hash_short'] = edges["tx_hash"].str[:10]
edges = edges.drop(columns=['tx_hash'])

print(f"total_edges={len(edges)}")
print("top_5_destinations:")
print(edges['to_address'].value_counts().head(5))
print("top_5_senders:")
print(edges["from_address"].value_counts().head(5))

edges = edges[["from_address","to_address","tx_hash_short","label","value_or_token"]]

total_edges=15
top_5_destinations:
to_address
0xd90e2f925DA726b50C4Ed8D0Fb90Ad053324F31b    14
0x79224bc0bf70ec34f0ef56ed8251619499a59def     1
Name: count, dtype: int64
top_5_senders:
from_address
0x1c5dCdd006EA78a7E4783f9e6021C32935a10fb4    15
Name: count, dtype: int64
